In [10]:
from fastapi import FastAPI
from langserve import add_routes
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langgraph.graph import StateGraph , START,END, MessagesState
from typing import TypedDict


llm = ChatOllama(
    model="qwen3.5:9b",
    temperature=0
)

 

In [11]:
class ModerationState(TypedDict):
    post_content: str
    user_reputation: str
    formatted_post: str
    content_flag: str
    result: str
    
    


# Analyze post

In [12]:
def analyze_content(state:ModerationState)->ModerationState:
    content = state['post_content'].lower()
    if 'spam' in content or 'scam' in content:
        flag = 'rejected'
    elif state['user_reputation'] == 'new_user':
        flag = 'review'
    else:
        flag = 'approved' 
    return {'content_flag':flag}           

In [13]:
def format_post(state: ModerationState)->ModerationState:
    formatted = f"user ({state['user_reputation']}) says: {state['post_content']}"
    return {"formatted_post":formatted}

In [14]:
def approve_post(state: ModerationState)->ModerationState:
    result =  " Post successfully published to the timeline"
    return {"result":result}
def review_post(state: ModerationState)->ModerationState:
    result =  " Post reviewed  to the timeline"
    return {"result":result}

def reject_post(state: ModerationState)->ModerationState:
    result =  " Post deleted from the timeline"
    return {"result":result}

## Check post

In [15]:
def check_post(state: ModerationState)->ModerationState:
    status = state['content_flag']
    print(status)
    
    if state['content_flag'] == 'rejected':
        print('rejected xxxxxxxxx')
        return 'reject_post'
    elif state['content_flag'] == 'review':
        print('review xxxxxxxxx')
        return 'review_post'
    else:
        return 'approve_post'

## Define graph

In [16]:
graph  = StateGraph(ModerationState)

graph.add_node('format_post',format_post)
graph.add_node('analyze_content',analyze_content)
graph.add_node('approve_post',approve_post)
graph.add_node('review_post',review_post)
graph.add_node('reject_post',reject_post)
graph.add_edge(START,'format_post')
graph.add_edge('format_post','analyze_content')
graph.add_conditional_edges('analyze_content',check_post)
graph.add_edge('reject_post',END)
graph.add_edge('review_post',END)
graph.add_edge('approve_post',END)

 
workflow = graph.compile()

In [20]:
ininit_state = {
   "post_content": 'what is temperature transmitter  ',
    "user_reputation": ""
 
}
workflow.invoke(ininit_state)

approved


{'post_content': 'what is temperature transmitter  ',
 'user_reputation': '',
 'formatted_post': 'user () says: what is temperature transmitter  ',
 'content_flag': 'approved',
 'result': ' Post successfully published to the timeline'}